# Designed against measured ratios in a TMT spike-in experiment

**Data kit lab** `tmt-spike-in-ratio-compression` · Discovery · about 30 minutes

**The question.** Do the four proteins spiked into an Erwinia background show their designed abundance ratios across the six TMT reporter channels, and how much are the measured ratios compressed towards 1?

**What you do.**

1. Read an mzTab file line by line and find its six reporter channels.
2. Sort the peptide rows into the background and the four spiked proteins.
3. Compute each protein's channel ratios from its peptides.
4. Compare them with the designed ratios and measure the compression.

**The data.** Source: PRIDE Archive, EMBL-EBI, project PXD000001 (submitted by Gatto and Christoforou), under the EMBL-EBI terms of use: attribute PRIDE and the submitters. Nothing is copied into this notebook: the first cell downloads the file from PRIDE (about 0.3 MB). The raw spectra (about 243 MB) are not needed.

**What you compare with.** The project record prints the ratios the experiment was *designed* to have. They are intentions, not measurements. No measured ratios are published with the record.

**You need.** Python 3.10 or later with `pandas` and `numpy`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'tmt-spike-in-ratio-compression'
FILES = json.loads(r'''{
 "mztab": {
  "url": "https://ftp.pride.ebi.ac.uk/pride/data/archive/2012/03/PXD000001/F063721.dat-mztab.txt",
  "name": "mztab.txt",
  "sha256": "5993f06ba9c90a46b2aa661d52cbac762736bcb95b788917e240df3bbc6ae691",
  "bytes": 304798
 }
}''')
PUBLISHED = json.loads(r'''{
 "erwinia_r127": {
  "label": "Erwinia background peptides: channel 127 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "erwinia_r128": {
  "label": "Erwinia background peptides: channel 128 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "erwinia_r129": {
  "label": "Erwinia background peptides: channel 129 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "erwinia_r130": {
  "label": "Erwinia background peptides: channel 130 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "erwinia_r131": {
  "label": "Erwinia background peptides: channel 131 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "enolase_r126": {
  "label": "Enolase spike (P00924, ENO1_YEAST): channel 126 / channel 129",
  "value": 10.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "enolase_r127": {
  "label": "Enolase spike (P00924, ENO1_YEAST): channel 127 / channel 129",
  "value": 5.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "enolase_r128": {
  "label": "Enolase spike (P00924, ENO1_YEAST): channel 128 / channel 129",
  "value": 2.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "enolase_r130": {
  "label": "Enolase spike (P00924, ENO1_YEAST): channel 130 / channel 129",
  "value": 2.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "enolase_r131": {
  "label": "Enolase spike (P00924, ENO1_YEAST): channel 131 / channel 129",
  "value": 10.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "bsa_r127": {
  "label": "BSA spike (P02769, ALBU_BOVIN): channel 127 / channel 126",
  "value": 2.5,
  "tolerance": {
   "abs": 0.05
  }
 },
 "bsa_r128": {
  "label": "BSA spike (P02769, ALBU_BOVIN): channel 128 / channel 126",
  "value": 5.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "bsa_r129": {
  "label": "BSA spike (P02769, ALBU_BOVIN): channel 129 / channel 126",
  "value": 10.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "bsa_r130": {
  "label": "BSA spike (P02769, ALBU_BOVIN): channel 130 / channel 126",
  "value": 5.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "bsa_r131": {
  "label": "BSA spike (P02769, ALBU_BOVIN): channel 131 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "phosb_r126": {
  "label": "Phosphorylase B spike (P00489, PYGM_RABIT): channel 126 / channel 130",
  "value": 2.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "phosb_r127": {
  "label": "Phosphorylase B spike (P00489, PYGM_RABIT): channel 127 / channel 130",
  "value": 2.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "phosb_r128": {
  "label": "Phosphorylase B spike (P00489, PYGM_RABIT): channel 128 / channel 130",
  "value": 2.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "phosb_r129": {
  "label": "Phosphorylase B spike (P00489, PYGM_RABIT): channel 129 / channel 130",
  "value": 2.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "phosb_r131": {
  "label": "Phosphorylase B spike (P00489, PYGM_RABIT): channel 131 / channel 130",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "cytc_r127": {
  "label": "Cytochrome C spike (P62894, CYC_BOVIN): channel 127 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "cytc_r128": {
  "label": "Cytochrome C spike (P62894, CYC_BOVIN): channel 128 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "cytc_r129": {
  "label": "Cytochrome C spike (P62894, CYC_BOVIN): channel 129 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "cytc_r130": {
  "label": "Cytochrome C spike (P62894, CYC_BOVIN): channel 130 / channel 126",
  "value": 1.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "cytc_r131": {
  "label": "Cytochrome C spike (P62894, CYC_BOVIN): channel 131 / channel 126",
  "value": 2.0,
  "tolerance": {
   "abs": 0.5
  }
 }
}''')
PUBLISHED_FROM = 'PRIDE Archive API record PXD000001, field projectDescription: designed reporter ion ratios, read in channel order 126 to 131 | https://www.ebi.ac.uk/pride/ws/archive/v2/projects/PXD000001'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Read the file by line prefix

An mzTab file mixes several tables in one text file. The first field of each line says what it is: `MTD` metadata, `PEH` the header of the peptide table, `PEP` a peptide row. This file predates mzTab 1.0, so read it as plain text.

In [ ]:
import numpy as np
import pandas as pd

lines = [l.rstrip("\n").split("\t") for l in data_file("mztab").read_text(encoding="utf-8").splitlines() if l.strip()]
print(pd.Series([l[0] for l in lines]).value_counts().to_string())
for l in lines:
    if l[0] == "MTD":
        print(" | ".join(l[1:]))

The metadata maps the six abundance columns `peptide_abundance_sub[1]` to `[6]` to TMT reagents 126 to 131, and says the values are counts.

In [ ]:
header = next(l for l in lines if l[0] == "PEH")[1:]
pep = pd.DataFrame([l[1:] for l in lines if l[0] == "PEP"], columns=header)
CHANNELS = ["126", "127", "128", "129", "130", "131"]
ab_cols = [f"peptide_abundance_sub[{i}]" for i in range(1, 7)]
print(pep.shape)
pep[["sequence", "accession", "database", "charge"] + ab_cols].head()

## 2. Background and spikes

Each spike is named in the project record by its UniProt accession. A row belongs to a spike when its accession field contains that accession; every other row is background.

In [ ]:
SPIKES = {"enolase": "P00924", "bsa": "P02769", "phosb": "P00489", "cytc": "P62894"}

def group_of(acc):
    for name, uniprot in SPIKES.items():
        if uniprot in acc:
            return name
    return "erwinia"

pep["group"] = pep["accession"].map(group_of)
print(pep["group"].value_counts().to_string())
for name in SPIKES:
    assert (pep["group"] == name).any(), f"no rows for {name}"
    print(name, sorted(pep.loc[pep["group"] == name, "accession"].unique()))
print("background databases:", pep.loc[pep["group"] == "erwinia", "database"].value_counts().to_dict())

Keep rows with all six reporter values present and above zero.

In [ ]:
values = pep[ab_cols].apply(pd.to_numeric, errors="coerce")
values.columns = CHANNELS
keep = values.notna().all(axis=1) & (values > 0).all(axis=1)
print("rows kept per group:")
print(pd.DataFrame({"all": pep["group"].value_counts(), "kept": pep.loc[keep, "group"].value_counts()}).fillna(0).astype(int))
values, groups = values[keep], pep.loc[keep, "group"]

## 3. From peptides to a protein ratio

The design gives each protein a channel at 1, but not the same one. Divide each row by that protein's reference channel, then take the median over the protein's rows for each channel. The median keeps a few very intense or noisy peptides from dominating; each row counts once.

In [ ]:
DESIGN_REF = {"erwinia": "126", "enolase": "129", "bsa": "126", "phosb": "130", "cytc": "126"}
medians = {}
for name, ref in DESIGN_REF.items():
    rows = values[groups == name]
    medians[name] = (rows.div(rows[ref], axis=0)).median()
measured = pd.DataFrame(medians).T[CHANNELS]
measured.round(2)

## 4. Designed against measured

`PUBLISHED` holds the designed ratios, copied by code from the project record. Put them beside the measurements.

In [ ]:
designed = pd.DataFrame(np.nan, index=measured.index, columns=CHANNELS)
for name, ref in DESIGN_REF.items():
    designed.loc[name, ref] = 1.0
for key, check in PUBLISHED.items():
    name, ch = key.split("_r")
    designed.loc[name, ch] = check["value"]
pd.concat({"designed": designed, "measured": measured.round(2)}, axis=1)

## 5. How much compression?

Ratio compression pulls every ratio towards 1. One way to see it: for each spike, compare the spread of the measured log2 ratios across the six channels (highest over lowest) with the designed spread. A value of 1 means no compression. Shown, not checked.

In [ ]:
comp = {}
for name in SPIKES:
    d, m = np.log2(designed.loc[name].astype(float)), np.log2(measured.loc[name].astype(float))
    comp[name] = {"designed_log2_range": d.max() - d.min(), "measured_log2_range": m.max() - m.min(),
                  "fraction_of_design": (m.max() - m.min()) / (d.max() - d.min())}
pd.DataFrame(comp).T.round(2)

The background should be flat (1:1:1:1:1:1). Its measured ratios show how evenly the channels were loaded. Dividing each spike's ratios by the background's is a common correction; it is shown here, not checked.

In [ ]:
background = measured.loc["erwinia"]
corrected = {}
for name, ref in DESIGN_REF.items():
    if name == "erwinia":
        continue
    corrected[name] = measured.loc[name] / (background / background[ref])
pd.DataFrame(corrected).T[CHANNELS].round(2)

## 6. Check against the design

`report` prints each measured ratio beside its designed value. The tolerance is half a unit of the last printed digit of the design number: 0.5 for 10, 5, 2 and 1, and 0.05 for 2.5.

In [ ]:
RESULTS = {}
for key in PUBLISHED:
    name, ch = key.split("_r")
    RESULTS[key] = float(measured.loc[name, ch])
report(RESULTS)

## What the result says

Look at the large designed ratios (10 for enolase and BSA). Do the measured values reach them? If they move in the designed direction but fall short, that is ratio compression: in an MS2-level TMT measurement, other peptides isolated with the target add reporter ions to every channel. Ratios of 1 and 2 are easier to meet within 0.5, which says as much about the tolerance as about the data.

## Try next

1. Take the mean instead of the median in section 3. Which proteins change most?
2. Use channel 126 as the reference for every protein. What does the enolase design look like then?
3. The file name says `isol2`: the precursor isolation width. Read about how isolation width affects co-isolation, and predict whether a narrower window would raise or lower the measured enolase ratio.
4. Count how many distinct peptide sequences each spike has. Are the proteins with few peptides the noisiest?